# Map:
[here](./map.html)!


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from shapely.geometry import Point, Polygon
import folium
from folium import Element
import ipywidgets as widgets
from IPython.display import display
from IPython.display import clear_output
from base64 import b64encode

In [2]:
total_df = pd.read_csv('data/total_df.csv')
# Extract the first row for each cast (based on File Name)
cast_locations = total_df.groupby("File Name").first().reset_index()
cast_locations = cast_locations[["Latitude", "Longitude", "File Name", "TestType", "Area"]]
image_df = pd.read_csv('data/profile_data.csv')
cast_locations['ImagePath'] = image_df['ImagePath']
cast_locations['Date'] = pd.to_datetime(cast_locations['File Name'].str.extract(r'_(\d{8})_')[0], format='%Y%m%d')
cast_locations.head()

,Latitude,Longitude,File Name,TestType,Area,ImagePath,Date
0,40.274148,-106.839900,CC2435009_20250513_181211,cast,MOR,data/profiles\profile_CC2435009_20250513_18121...,2025-05-13
1,40.276507,-106.841752,CC2435009_20250513_181617,cast,MOR,data/profiles\profile_CC2435009_20250513_18161...,2025-05-13
2,40.279905,-106.851733,CC2435009_20250513_182530,cast,MID,data/profiles\profile_CC2435009_20250513_18253...,2025-05-13
3,40.279532,-106.852019,CC2435009_20250513_182851,cast,MID,data/profiles\profile_CC2435009_20250513_18285...,2025-05-13
4,40.283628,-106.853450,CC2435009_20250513_194404,cast,DOC,data/profiles\profile_CC2435009_20250513_19440...,2025-05-13


In [4]:
# ---- Color Map for Areas ----
area_colors = {
    'DOC': 'red',
    'HAR': 'cyan',
    'MOR': 'purple',
    'GLZ': 'orange',
    'KEY': 'darkblue',
    'LIL': 'darkred',
    'ASP': 'pink',
    'MID': 'gray'
}

# ---- Legend HTML ----
legend_html = """
<div style="
    position: fixed; 
    bottom: 30px; left: 30px; width: 150px; z-index:500;
    background-color: white; padding: 10px; border: 2px solid grey;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.3); font-size:14px;">
<b>Areas</b><br>
<i style='background:red; width:10px; height:10px; float:left; margin-right:8px; display:inline-block'></i>Dock<br>
<i style='background:cyan; width:10px; height:10px; float:left; margin-right:8px; display:inline-block'></i>Harding Cove<br>
<i style='background:purple; width:10px; height:10px; float:left; margin-right:8px; display:inline-block'></i>Morrison Cove<br>
<i style='background:orange; width:10px; height:10px; float:left; margin-right:8px; display:inline-block'></i>Glizzy Cove<br>
<i style='background:darkblue; width:10px; height:10px; float:left; margin-right:8px; display:inline-block'></i>Keystone Cove<br>
<i style='background:darkred; width:10px; height:10px; float:left; margin-right:8px; display:inline-block'></i>Small Inlet<br>
<i style='background:pink; width:10px; height:10px; float:left; margin-right:8px; display:inline-block'></i>Aspen View<br>
<i style='background:gray; width:10px; height:10px; float:left; margin-right:8px; display:inline-block'></i>Middle of Reservoir
</div>
"""

# ---- Widget Setup ----
#date_picker = widgets.DatePicker(description='Pick a date', value=cast_locations['Date'].min())
#toggle_button = widgets.ToggleButton(value=True, description="Show All Data", button_style='info')
map_output = widgets.Output()

def update_map(change=None):
    with map_output:
        clear_output(wait=True)
        #if toggle_button.value:
        filtered = cast_locations
        #else:
            #pass
            #selected_date = date_picker.value
            #filtered = cast_locations[cast_locations['Date'] == pd.to_datetime(selected_date)]
        
        # Create map
        m = folium.Map(
            location=[filtered["Latitude"].mean(), filtered["Longitude"].mean()],
            zoom_start=13
        )

        # ---- Plot Polygons ----
#        for name, coords in polygon_coords.items():
#            folium.Polygon(
#                locations=coords,
#                color=area_colors[name],
#                fill=True,
#                fill_opacity=0.15,
#                popup=f"{name} Area"
#            ).add_to(m)
        
        # Plot points
        for _, row in filtered.iterrows():

            if row['TestType'] == 'point':
                # Show raw measurement values
                point = total_df[total_df['File Name'] == row['File Name']]
                popup_html = f"""
                <b>File:</b> {row['File Name']}<br>
                <b>Test Type:</b> {'Point'}<br>
                <b>Lat:</b> {row['Latitude']}<br>
                <b>Lon:</b> {row['Longitude']}<br>
                <b>Area:</b> {row['Area']}<br>
                <b>Temperature (°C):</b> {point['Temperature (°C)'].iloc[0]}<br>
                <b>Conductivity (µS/cm):</b> {point['Conductivity (µS/cm)'].iloc[0]}<br>
                <b>Salinity (PSS):</b> {point['Salinity (PSS)'].iloc[0]}<br>
                <b>Density (kg/m³):</b> {point['Density (kg/m³)'].iloc[0]}<br>
                """
                iframe = folium.IFrame(popup_html, width=200, height=200)
                popup = folium.Popup(iframe, max_width=200)   

            elif row['TestType'] == 'cast':
                with open(row['ImagePath'], 'rb') as f:
                    encoded = b64encode(f.read()).decode()
                popup_html = f"""
                    <div>
                        <a href="data:image/png;base64,{encoded}" target="_blank">
                            <img src="data:image/png;base64,{encoded}" width="700">
                        </a><br>
                        <b>Date:</b> {row['Date'].date()}<br>
                        <b>Test Type:</b> {'Cast'}<br>
                        <b>File:</b> {row['File Name']}<br>
                        <b>Lat:</b> {row['Latitude']}<br>
                        <b>Lon:</b> {row['Longitude']}<br>
                        <b>Area:</b> {row['Area']}
                    </div>
                """
                iframe = folium.IFrame(html=popup_html, width=250, height=250)
                popup = folium.Popup(iframe, max_width=250)    
            
            folium.CircleMarker(
                location=[row["Latitude"], row["Longitude"]],
                radius=5,
                popup = popup,
                color=area_colors.get(row["Area"], "gray"),
                fill=True,
                fill_color=area_colors.get(row["Area"], "gray"),
                fill_opacity=0.9
            ).add_to(m)

        m.get_root().html.add_child(Element(legend_html))
        display(m)
        m.save("map.html")

# ---- Connect Handlers ----
#date_picker.observe(update_map, names='value')
#toggle_button.observe(update_map, names='value')

# ---- Show UI ----
#display(widgets.HBox([toggle_button]))
display(map_output)
update_map()  # Initial call




Output()